In [1]:
# -*- coding: utf-8 -*-
from Library import utils, dataset
import os
from tensorflow import keras
from tqdm import tqdm
import numpy as np
from datetime import datetime
import logging
import matplotlib.pyplot as plt

# Fungsi untuk membuat diagram batang metrik komprehensif
def plot_comprehensive_metrics(metrics, save_path):
    # Mapping nama label ke key yang tepat sesuai output Anda
    labels = ['Accuracy', 'Recall', 'Precision', 'FPR', 'F1-Score']
    keys = [
        'Accuracy (avg.)', 
        'True positive rate (avg.)', 
        'Positive predictive value (avg.)', 
        'False positive rate (avg.)', 
        'F1-score (avg.)'
    ]
    
    # Mengambil nilai berdasarkan key yang tepat
    values = [metrics.get(k, 0) for k in keys]
    
    print(f"[DEBUG] Nilai yang akan diplot: {values}") 

    plt.figure(figsize=(12, 6))
    bars = plt.bar(labels, values, color=['#2c3e50', '#3498db', '#e67e22', '#e74c3c', '#27ae60'])
    
    plt.ylim(0, 1.0)
    plt.ylabel('Score')
    plt.title('Performance Metrics Evaluation (Average)')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Tambahkan angka di atas batang
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom')
                
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"[INFO] Grafik metrik komprehensif tersimpan di: {save_path}")

# Main execution
if __name__ == "__main__":

    #===================================================================
    #                    1. KONFIGURASI PATH STEAD 5000 3C
    #===================================================================
    KEY_DATA_DIR = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000'
    KEY_TEST_FILE = 'STEAD_5000_3C_20260719_062844.json'  # File 3C hasil ekstraksi
    DATA_TAG = "STEAD_5000_DOMAIN"
    MODEL_TAG = "MCU_Quake_3C"
    INPUT_WIN = 7 
    SAMPLING_RATE = 100
    
    # Label di JSON: 'ev' = gempa, 'no' = noise
    true_labels = ["NO", "LE"]  # untuk plot confusion matrix
    source_to_code = {"no": 0, "ev": 1}  # mapping label STEAD ke kode biner

    # Path Model dan Embedding 
    BASE_REP = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mulai_juli/mcquake_ori_file/Code & Figure demo"
    MODEL_PATH = os.path.join(BASE_REP, "Pre-trained model/MCU-Quake 5-20")
    
    # =====================================================================
    # PILIH EMBEDDING REFERENSI:
    # Opsi 1: UUSS (digunakan di paper, baseline)
    # Opsi 2: STEAD (lebih fair karena data testing juga dari STEAD)
    # =====================================================================
    #EMB_DIR = os.path.join(BASE_REP, "Typical embedding/Embedding_data train 3C, STEAD norm7 mag3 L n61099, 30172538")
    EMB_DIR = os.path.join(BASE_REP, "Typical embedding/Embedding_data train 3C, UUSS n11275 std15, 30120909")

    #===================================================================
    #                           2. PREPARE FILES
    #===================================================================
    print(f"[INFO] Memuat Dataset STEAD 5000: {KEY_TEST_FILE}")
    test_data = dataset.load_json_data(os.path.join(KEY_DATA_DIR, KEY_TEST_FILE))
    
    SAVE_BASE = '/Volumes/Extreme SSD/stream_stead/data_stead/benchmark_stead_5000_3c'
    now = datetime.now()
    time_str = now.strftime("%d%H%M%S")
    save_dir = os.path.join(SAVE_BASE, f"{MODEL_TAG}_{DATA_TAG}_{time_str}")
    if not os.path.exists(save_dir): 
        os.makedirs(save_dir)

    # Logger
    log_file_path = os.path.join(save_dir, "task_log_stead_5000_3c.txt")
    logging.basicConfig(filename=log_file_path, level=logging.INFO, filemode='w',
                        format='%(asctime)s - [%(levelname)s]: %(message)s')
    logger = logging.getLogger()
    logger.addHandler(logging.StreamHandler())

    # Load Model & Embeddings
    embedding_model = keras.models.load_model(filepath=MODEL_PATH)
    embedding_Z = dataset.load_embedding_data(EMB_DIR, "Embedding data, Z.json")
    embedding_N = dataset.load_embedding_data(EMB_DIR, "Embedding data, N.json")
    embedding_E = dataset.load_embedding_data(EMB_DIR, "Embedding data, E.json")

    logger.info(f"Estimating STEAD 5000 embeddings statistics (KDE) using UUSS reference...")
    embeddings_3C_PDFs = utils.embedding_PDFs_3D(embedding_Z, embedding_N, embedding_E)

    #===================================================================
    #                            3. EVALUATION 3C
    #===================================================================
    total_true_3C, total_pred_3C = [], []
    num_points = int(INPUT_WIN * SAMPLING_RATE)
    keys_list = list(test_data.keys())

    logger.info(f"Memproses {len(keys_list)} data STEAD 5000 (3C)...")

    for i in tqdm(range(len(keys_list)), desc="Inference STEAD 5000 3C"):
        record_key = keys_list[i]
        record = test_data[record_key]
        true_label = record["type"]  # 'ev' atau 'no'
        
        try:
            # Sinyal & Noise 3C
            Z_n, N_n, E_n = record["Z_noise"][-num_points:], record["N_noise"][-num_points:], record["E_noise"][-num_points:]
            Z_s, N_s, E_s = record["Z"][:num_points], record["N"][:num_points], record["E"][:num_points]

            # Embeddings untuk 3 komponen
            _in_Zn, _in_Nn, _in_En = utils.latent_codes_1D(Z_n, embedding_model), utils.latent_codes_1D(N_n, embedding_model), utils.latent_codes_1D(E_n, embedding_model)
            _in_Zs, _in_Ns, _in_Es = utils.latent_codes_1D(Z_s, embedding_model), utils.latent_codes_1D(N_s, embedding_model), utils.latent_codes_1D(E_s, embedding_model)

            # Inferensi 3C KDE - Noise
            emb_n_3c = np.array([_in_En, _in_Nn, _in_Zn]).reshape(1,-1)
            p_n_3c, _, _ = utils.infer_3C_PDFs(emb_n_3c, embeddings_3C_PDFs, "Kernel")
            
            # Inferensi 3C KDE - Earthquake (Signal)
            emb_s_3c = np.array([_in_Es, _in_Ns, _in_Zs]).reshape(1,-1)
            p_s_3c, _, _ = utils.infer_3C_PDFs(emb_s_3c, embeddings_3C_PDFs, "Kernel")

            # Mapping True Label: noise window dianggap 0, signal window dianggap 1
            # true_label: 'no' -> 0, 'ev' -> 1
            # total_true_3C: [0, 1] selalu (noise window harus 0, signal window harus 1)
            total_true_3C.extend([0, 1]) 
            # Prediksi: threshold 1.0
            total_pred_3C.extend([1 if p_n_3c >= 1 else 0, 1 if p_s_3c >= 1 else 0])
            
        except Exception as e:
            logger.error(f"Error pada {record_key}: {e}")
            continue

    #===================================================================
    #                        4. METRICS & SAVE
    #===================================================================
    matrix_3C, metrics_3C = utils.calc_confusion_metrics(total_true_3C, total_pred_3C)
    
    # Visualisasi Confusion Matrix
    fig_3C = utils.plot_confusion(f"{MODEL_TAG} {DATA_TAG} 3C", true_labels, matrix_3C, metrics_3C)
    fig_3C.savefig(os.path.join(save_dir, "STEAD_5000_3C_confusion.jpg"), dpi=300)
    
    # Simpan metrics ke JSON
    dataset.save_json_data(os.path.join(save_dir, "stead_5000_3C_metrics.json"), metrics_3C)
    
    # Plot comprehensive metrics (bar chart)
    plot_comprehensive_metrics(metrics_3C, os.path.join(save_dir, "stead_5000_3C_metrics_bar.jpg"))

    # =================================================================
    #                     5. HASIL AKHIR
    # =================================================================
    logger.info("\n" + "="*40)
    logger.info(f"STEAD 5000 3C ACCURACY: {metrics_3C.get('accuracy (avg.)')}")
    logger.info(f"STEAD 5000 3C F1-SCORE: {metrics_3C.get('f1-score (avg.)')}")
    logger.info("="*40)
    logger.info("Pengujian STEAD_5000_3C Selesai.")

[INFO] Memuat Dataset STEAD 5000: STEAD_5000_3C_20260719_062844.json


2026-07-19 14:59:07.840828: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
2026-07-19 14:59:07.840874: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
2026-07-19 14:59:07.840891: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.66 GB
2026-07-19 14:59:07.840932: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-07-19 14:59:07.840949: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


No training configuration found in save file, so the model was *not* compiled. Compile it manually.
Estimating STEAD 5000 embeddings statistics (KDE) using UUSS reference...
Memproses 5000 data STEAD 5000 (3C)...
Inference STEAD 5000 3C:   4%|▍         | 196/5000 [00:04<01:48, 44.17it/s]


KeyboardInterrupt: 